# Generate On-Policy DPO Pairs
Generates preference pairs using **your own SFT model** + DeepSeek as judge.

**How it works:**
1. Load 10,000 prompts from your Alpaca SFT dataset
2. Generate 4 candidate responses per prompt using the SFT checkpoint
3. Send each prompt + 4 candidates to DeepSeek to rank best → worst
4. Take rank 1 (chosen) and rank 4 (rejected) as the preference pair

**Why on-policy?** Both chosen and rejected come from your model's own distribution —
no capability gap, no length exploitation, no GPT-4 imitation.

**Resumable:** Generations and judgments are saved incrementally.

**Prerequisites:** SFT checkpoint + Alpaca dataset on Drive.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tokenizers openai

In [ ]:
import os, json, time, random, logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from openai import OpenAI
from google.colab import drive, userdata

torch._logging.set_logs(inductor=logging.WARNING, dynamic=logging.WARNING)
logging.getLogger("torch._inductor").setLevel(logging.WARNING)

drive.mount('/content/drive')

# ==================== CONFIGURATION ====================
SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
CHECKPOINT_DIR = os.path.join(SPARKYLLM, "checkpoints")
SFT_CKPT = os.path.join(CHECKPOINT_DIR, "gpt_medium_sft_best.pth")
TOKENIZER_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "tokenizer.json")
ALPACA_RAW = os.path.join(SPARKYLLM, "datasets_sft", "alpaca", "alpaca_raw.jsonl")
SFT_META = os.path.join(SPARKYLLM, "datasets_sft", "alpaca", "tokenize_meta.json")

# Output
OUT_DIR = os.path.join(SPARKYLLM, "datasets_dpo", "on_policy")
GENERATIONS_FILE = os.path.join(OUT_DIR, "generations.jsonl")       # intermediate
RAW_FILE = os.path.join(OUT_DIR, "on_policy_raw.jsonl")             # final output
META_FILE = os.path.join(OUT_DIR, "meta.json")
os.makedirs(OUT_DIR, exist_ok=True)

# Generation
NUM_PROMPTS = 10000
CANDIDATES_PER_PROMPT = 4
TEMPERATURE = 0.9
TOP_K = 50
MAX_NEW_TOKENS = 200

# Judge — load API key from Colab Secrets (Settings → Secrets → DEEPSEEK_API_KEY)
DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
assert DEEPSEEK_API_KEY, "DEEPSEEK_API_KEY not found in Colab Secrets"
JUDGE_MODEL = "deepseek-chat"

# Hardware
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda")

# Verify paths
assert os.path.exists(SFT_CKPT), f"SFT checkpoint not found: {SFT_CKPT}"
assert os.path.exists(ALPACA_RAW), f"Alpaca data not found: {ALPACA_RAW}"
print("All paths verified")

In [ ]:
# ==================== MODEL DEFINITION ====================

BLOCK_SIZE = 768
EMBED_DIM = 1280
NUM_LAYERS = 24
NUM_HEADS = EMBED_DIM // 64  # 20
FF_HIDDEN_DIM = 4 * EMBED_DIM  # 5120

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        self.c_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2)
        self.w2 = nn.Linear(hidden_dim, embed_dim)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x + F.dropout(self.attn(self.norm1(x)), p=self.dropout, training=self.training)
        x = x + F.dropout(self.ff(self.norm2(x)), p=self.dropout, training=self.training)
        return x

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, EMBED_DIM)
        self.blocks = nn.ModuleList(
            [TransformerBlock(EMBED_DIM, NUM_HEADS, FF_HIDDEN_DIM, 0.1) for _ in range(NUM_LAYERS)]
        )
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

print("Model definition ready")

In [ ]:
# ==================== LOAD MODEL + TOKENIZER ====================

# Tokenizer
!cp "{TOKENIZER_PATH}" /content/tokenizer.json
tokenizer = Tokenizer.from_file("/content/tokenizer.json")

# Vocab
with open(SFT_META, "r") as f:
    tok_meta = json.load(f)
VOCAB_SIZE = int(tok_meta["vocab_size"])
EOT_ID = int(tok_meta["eot_id"])
if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
print(f"Vocab: {VOCAB_SIZE} | EOT: {EOT_ID}")

# Model
print(f"Loading SFT model from {SFT_CKPT}...")
model = SimpleGPT(VOCAB_SIZE).to(device)
state_dict = torch.load(SFT_CKPT, map_location=device)
clean_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model_state = model.state_dict()
safe_state = {k: v for k, v in clean_state.items() if k in model_state and v.shape == model_state[k].shape}
model.load_state_dict(safe_state, strict=False)
model.eval()
del state_dict, clean_state, safe_state

print("Compiling model...")
model = torch.compile(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params / 1e6:.0f}M parameters")

In [ ]:
# ==================== LOAD PROMPTS ====================

def format_prompt(obj):
    """Format an Alpaca example into the prompt string (same as SFT adapter)."""
    parts = [f"Instruction: {obj['instruction']}"]
    if obj.get("input"):
        parts.append(f"Input: {obj['input']}")
    parts.append("Response:")
    return "\n".join(parts) + " "

# Load all prompts
all_prompts = []
with open(ALPACA_RAW, "r") as f:
    for line in f:
        obj = json.loads(line)
        prompt = format_prompt(obj)
        # Skip prompts that are too long (leave room for response)
        prompt_ids = tokenizer.encode(prompt).ids
        if len(prompt_ids) < BLOCK_SIZE - MAX_NEW_TOKENS:
            all_prompts.append({"prompt_text": prompt, "prompt_len": len(prompt_ids)})

print(f"Loaded {len(all_prompts):,} prompts (after length filter)")

# Sample NUM_PROMPTS
random.seed(42)
if len(all_prompts) > NUM_PROMPTS:
    prompts = random.sample(all_prompts, NUM_PROMPTS)
else:
    prompts = all_prompts
print(f"Selected {len(prompts):,} prompts for generation")

In [ ]:
# ==================== PHASE 1: GENERATE CANDIDATES (BATCHED) ====================
# Generates CANDIDATES_PER_PROMPT responses per prompt.
# All 4 candidates are generated in a single batch for ~3-4x speedup.
# Resumable: skips prompts already in generations.jsonl.

@torch.no_grad()
def generate_batch(prompt_text, num_candidates, temperature, top_k, max_new_tokens):
    """Generate num_candidates responses in parallel. Returns list of generated texts."""
    ids = tokenizer.encode(prompt_text).ids
    prompt_len = len(ids)

    # Seed each candidate differently via initial noise — we set per-candidate seeds below
    x = torch.tensor([ids] * num_candidates, dtype=torch.long, device=device)  # (B, T)
    finished = [False] * num_candidates

    for step in range(max_new_tokens):
        if all(finished):
            break

        x_cond = x[:, -BLOCK_SIZE:]
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x_cond)
        logits = logits[:, -1, :] / temperature  # (B, vocab)

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('inf')

        probs = F.softmax(logits, dim=-1)
        next_ids = torch.multinomial(probs, num_samples=1)  # (B, 1)

        # Replace EOT with padding for finished sequences
        for i in range(num_candidates):
            if finished[i]:
                next_ids[i] = EOT_ID
            elif next_ids[i].item() == EOT_ID:
                finished[i] = True
                next_ids[i] = EOT_ID

        x = torch.cat([x, next_ids], dim=1)

    # Decode each candidate
    results = []
    for i in range(num_candidates):
        gen_ids = x[i, prompt_len:].tolist()
        # Strip trailing EOT padding
        if EOT_ID in gen_ids:
            gen_ids = gen_ids[:gen_ids.index(EOT_ID)]
        results.append(tokenizer.decode(gen_ids))
    return results


# Check for existing generations (resume support)
existing = set()
if os.path.exists(GENERATIONS_FILE):
    with open(GENERATIONS_FILE, "r") as f:
        for line in f:
            obj = json.loads(line)
            existing.add(obj["prompt_idx"])
    print(f"Resuming: {len(existing):,} prompts already generated")

# Warmup compile with batched generation
torch.manual_seed(0)
_ = generate_batch("Instruction: Hello.\nResponse: ", CANDIDATES_PER_PROMPT, TEMPERATURE, TOP_K, 10)

# Generate
t0 = time.time()
with open(GENERATIONS_FILE, "a") as f_out:
    for idx, p in enumerate(prompts):
        if idx in existing:
            continue

        # Set seed for reproducibility — same base seed as before
        torch.manual_seed(idx * 1000)

        candidates = generate_batch(
            p["prompt_text"], CANDIDATES_PER_PROMPT, TEMPERATURE, TOP_K, MAX_NEW_TOKENS
        )
        candidates = [c.strip() for c in candidates]

        obj = {
            "prompt_idx": idx,
            "prompt": p["prompt_text"],
            "candidates": candidates,
        }
        f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f_out.flush()

        if (idx + 1) % 100 == 0:
            elapsed = time.time() - t0
            rate = (idx + 1 - len(existing)) / elapsed * 60
            print(f"  {idx+1:,}/{len(prompts):,} | {rate:.0f} prompts/min | {elapsed/60:.1f} min elapsed")

elapsed = time.time() - t0
print(f"\nGeneration complete: {len(prompts):,} prompts x {CANDIDATES_PER_PROMPT} candidates")
print(f"Total time: {elapsed/60:.1f} minutes")

In [ ]:
# ==================== FREE GPU MEMORY ====================
# Model is no longer needed — free VRAM before the API judging phase.

del model
torch.cuda.empty_cache()
print("GPU memory freed")

In [ ]:
# ==================== PHASE 2: JUDGE WITH DEEPSEEK ====================
# Ranks the 4 candidates per prompt. Takes best and worst as the pair.
# Resumable: skips prompts already in the output file.

judge_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

JUDGE_SYSTEM = """You are an expert evaluator. You will see an instruction and 4 candidate responses from a small AI model.

Rank the responses from best (1) to worst (4) based on:
- Correctness and relevance to the instruction
- Coherence and fluency
- Conciseness (no unnecessary repetition or filler)

Respond ONLY with valid JSON in this exact format (no markdown, no explanation):
{"ranking": [best_idx, second_idx, third_idx, worst_idx]}

Where each index is 0, 1, 2, or 3 corresponding to Response 0-3. All four indices must appear exactly once."""


def judge_candidates(prompt, candidates, max_retries=3):
    """Call DeepSeek to rank 4 candidates. Returns ranking list or None on failure."""
    user_parts = [f"INSTRUCTION:\n{prompt}\n"]
    for i, cand in enumerate(candidates):
        user_parts.append(f"RESPONSE {i}:\n{cand}\n")
    user_msg = "\n".join(user_parts)

    for attempt in range(max_retries):
        try:
            completion = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=100,
            )
            raw = completion.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            result = json.loads(raw)
            ranking = result["ranking"]
            # Validate: must be a permutation of [0,1,2,3]
            assert sorted(ranking) == [0, 1, 2, 3], f"Invalid ranking: {ranking}"
            return ranking
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return None


# Load generations
generations = []
with open(GENERATIONS_FILE, "r") as f:
    for line in f:
        generations.append(json.loads(line))
print(f"Loaded {len(generations):,} generated prompts")

# Check for existing judgments (resume support)
judged_idxs = set()
if os.path.exists(RAW_FILE):
    with open(RAW_FILE, "r") as f:
        for line in f:
            obj = json.loads(line)
            judged_idxs.add(obj["prompt_idx"])
    print(f"Resuming: {len(judged_idxs):,} prompts already judged")

# Judge
t0 = time.time()
written = len(judged_idxs)
failed = 0

with open(RAW_FILE, "a") as f_out:
    for gen in generations:
        if gen["prompt_idx"] in judged_idxs:
            continue

        # Skip if any candidate is empty
        if any(len(c.strip()) == 0 for c in gen["candidates"]):
            failed += 1
            continue

        ranking = judge_candidates(gen["prompt"], gen["candidates"])

        if ranking is None:
            failed += 1
            continue

        best_idx = ranking[0]
        worst_idx = ranking[3]

        obj = {
            "prompt_idx": gen["prompt_idx"],
            "prompt": gen["prompt"],
            "chosen": gen["candidates"][best_idx],
            "rejected": gen["candidates"][worst_idx],
            "chosen_rank": 1,
            "rejected_rank": 4,
            "ranking": ranking,
        }
        f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f_out.flush()
        written += 1

        if written % 200 == 0:
            elapsed = time.time() - t0
            print(f"  {written:,} judged | {failed} failed | {elapsed/60:.1f} min")

        time.sleep(0.2)  # rate limit

elapsed = time.time() - t0
print(f"\nJudging complete!")
print(f"  {written:,} pairs written, {failed:,} failed")
print(f"  Time: {elapsed/60:.1f} minutes")

In [ ]:
# ==================== SAVE METADATA ====================

# Count final pairs
pair_count = 0
with open(RAW_FILE, "r") as f:
    for _ in f:
        pair_count += 1

meta = {
    "source": "on-policy (SFT model generations + DeepSeek judge)",
    "sft_checkpoint": os.path.basename(SFT_CKPT),
    "judge_model": JUDGE_MODEL,
    "num_prompts_sampled": len(prompts),
    "candidates_per_prompt": CANDIDATES_PER_PROMPT,
    "num_pairs": pair_count,
    "generation_config": {
        "temperature": TEMPERATURE,
        "top_k": TOP_K,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
}
with open(META_FILE, "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nDataset ready: {pair_count:,} on-policy preference pairs")
print(f"Saved to: {RAW_FILE}")
print(f"Meta: {META_FILE}")
print(f"\nNext step: run tokenize_on_policy.ipynb")